In [ ]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import re
import json
#from tqdm import tqdm
#import time
import os
from openai import OpenAI
import tiktoken

api_key = os.environ["OPENAI_API_KEY"]
org_key = os.environ.get("OPENAI_ORG_ID")

In [ ]:
df_elig = pd.read_csv('paper_collection/WoS_251031_eligible_design.csv')
df_pgg = pd.read_csv('input/pgg_validation_basePrompt.csv')

In [ ]:
#df_elig['collected_251031'] =  df_elig['file_id'].isna() # get status (old or new)

In [ ]:
df_elig['collected_251031'].sum()

In [ ]:
#df_elig.to_csv('paper_collection/WoS_251031_eligible.csv', index=False)

In [ ]:
df_elig.sort_values('Times Cited, All Databases', ascending=False, ignore_index=True).to_csv('paper_collection/WoS_251031_eligible.csv', index=False)

In [ ]:
df_elig.file_id.isna().sum()

In [ ]:
for i in range(50, 800, 50):
    print(i)

In [ ]:
client = OpenAI(api_key=api_key, organization=org_key) if org_key else OpenAI(api_key=api_key)

vector_store_id = "vs_68e867cb856881919afaf916060dcea8"

vector_store = client.vector_stores.retrieve(vector_store_id=vector_store_id)

In [ ]:
vector_store

Upload new file

In [ ]:
from openai import OpenAI
client = OpenAI()

client.files.create(
  file=open("mydata.jsonl", "rb"),
  purpose="fine-tune",
  expires_after={
    "anchor": "created_at",
    "seconds": 2592000
  }
)


In [ ]:
file_upload = client.files.create(
  file=open(df_elig.iloc[0]['file_path'], "rb"),
  purpose='assistants'
)

In [ ]:
file_upload

In [ ]:
file_upload.id

In [ ]:
df_elig['collected_251031'].sum()

In [ ]:
df_elig['file_id'].isna().sum()

In [ ]:
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        file_upload = client.files.create(
  file=open(row['file_path'], "rb"),
    purpose='assistants'
)
        row['file_id'] = file_upload.id # upload file id
        print(f"Upload: file_id {file_upload.id}")

In [ ]:
index_251031 = 0
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        df_elig.loc[i, 'file_id'] = script.split("\nUpload: file_id ")[1:][index_251031]
        index_251031 += 1

In [ ]:
df_elig.file_id.isna().sum()

In [ ]:
file_batches = list()
for i, row in df_elig[df_elig['collected_251031']].iterrows():
    file_batches.append(
        {
            "file_id": row["file_id"],
            "attributes": {"fileID": row["file_id"]}
        }
    )

In [ ]:
file_batches = list(df_elig[df_elig['collected_251031']]["file_id"])

In [ ]:
len(file_batches)

In [ ]:
def split_list_by_chunk_size(original_list, chunk_size=50):
    """
    Splits a list into sub-lists, each with a specified chunk size.

    Args:
        original_list (list): The list to be split.
        chunk_size (int): The maximum number of items in each sub-list.

    Returns:
        list: A list of sub-lists.
    """
    result_sublists = []
    for i in range(0, len(original_list), chunk_size):
        result_sublists.append(original_list[i:i + chunk_size])
    return result_sublists

In [ ]:
for l in split_list_by_chunk_size(file_batches, 50):
    vector_store_file_batch = client.vector_stores.file_batches.create_and_poll(
  vector_store_id=vector_store_id,
  file_ids=l
)
    print(vector_store_file_batch)

    

In [ ]:
# update file attributes
for file in df_elig[df_elig['collected_251031']]["file_id"]:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        attributes={"fileId": file}
    )


### Update chunking strategy

In [ ]:
# update file attributes
for file in df_elig["file_id"]:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        chunking_strategy={
        "type": "static",
        "max_chunk_size_tokens": 4096,
        "chunk_overlap_tokens": 500,
      }
    )


In [ ]:
vector_store_file_batch = client.vector_stores.file_batches.create(
  vector_store_id=vector_store_id,
  files=new_files
)
print(vector_store_file_batch)


In [ ]:
len(file_batches)

In [ ]:
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        client.vector_stores.files.create(
        vector_store_id=vector_store_id,
        file_id=row['file_id']
        )

In [ ]:
for fileid in script.split("\nUpload: file_id ")[1:]:
    

In [ ]:
df_elig['file_id'].isna().sum()

In [ ]:
# check upload status
for i, row in df_elig.iterrows():
    if row['collected_251031']:
        file = client.files.retrieve(
  row['file_id']
)
        print(file.status)
        #row['file_id'] = file_upload.id # upload file id
        #print(f"Upload: file_id {file_upload.id}")

In [ ]:
vector_store_file_batch = client.vector_stores.file_batches.create(
  vector_store_id=vector_store_id,
  files=new_files
)
print(vector_store_file_batch)


In [ ]:
for file in df_elig.file_id:
    client.vector_stores.files.update(
        vector_store_id=vector_store_id,
        file_id=file,
        attributes={"fileId": file}
    )


In [ ]:
client.vector_stores.files.list(
        vector_store_id=vector_store.id, limit=100
    ).data

In [ ]:
vector_store

In [ ]:
vector_store_files = client.vector_stores.files.list(
        vector_store_id=vector_store.id, limit=100, filter='failed'
    )
for file_obj in vector_store_files.data:
        if file_obj.status == "failed":
            print({"file_id": file_obj.id, "filename": file_obj.id})

In [ ]:
# note that this paper could not be parsed.
print(df_elig[df_elig.file_id == 'file-UV4X1AmP1CehypJPvbbdCicn'].file_path[55])

In [ ]:
client.vector_stores.files.delete(
    vector_store_id=vector_store.id,
    file_id='file-UV4X1AmP1CehypJPvbbdCicn'
)

In [ ]:
file_batch = client.vector_stores.file_batches.create_and_poll(
vector_store_id=vector_store.id, file_ids=['file-UV4X1AmP1CehypJPvbbdCicn']
)

In [ ]:
df_elig.shape

In [ ]:
file_batch = client.vector_stores.file_batches.create_and_poll(
vector_store_id=vector_store.id, file_ids=list(df_elig.file_id)[750:]
)

In [ ]:
client.vector_stores.retrieve(vector_store_id=vector_store.id)

In [ ]:
client = OpenAI(api_key=api_key, organization=org_key) if org_key else OpenAI(api_key=api_key)
vector_stores = client.vector_stores.list(limit=100)

In [ ]:
for store in vector_stores.data:
    print(f"Deleting vector store: {store.id}")
    deleted_store = client.vector_stores.delete(vector_store_id=store.id)
    print(f"Deleted: {deleted_store.id}")

In [ ]:
PATH_MAP    = "paper_collection/collection_mapping_251110.json"          # {collection: [paper-row idx]}
with open(PATH_MAP, encoding="utf-8") as fh:
    raw_map = json.load(fh)

#n_rows = len(df_d)
coll_map = {lab: [int(x) for x in raw if str(x).isdigit() and 0 <= int(x)]
            for lab, raw in raw_map.items()}

# RAG

In [ ]:
system_prompt = f"""We have conducted multiple public goods game experiments with varying experimental designs, to measure the effect of punishment in cooperative settings under various environments.
Your task is to predict how enabling a punishment mechanism to a specific game changes the ***efficiency*** compared to the same game with punishment disabled.
According to our experiments, whether punishment increases efficiency or not is highly dependent on a lot of dimensions in experiment design, and it is your job to navigate this heterogeneity and make accurate predictions.

***Efficiency*** is the ratio between the game players' behavior and that of a fully-cooperative group (i.e. a group in which all members contribute their full endowment in every round)
In other words, efficiency measures how close a group's total payoff is, compared to that of a group that always cooperates (i.e. always contributes the entire endowment, and benefits maximally from the multiplier). 
An efficiency value of 100% means that a group earned the same amount of coins as a hypothetical group that always cooperated.
            
For example, let's say a game has 5 players playing 10 rounds where 20 coins are given to each player per round and the multiplier for each contributed coin is 3.
In this case, the earning of a hypothetical "always cooperating" group is 5*10*20*3=3000 coins, while the earning of a hypothetical "never cooperating" group is 1000 coins.
Hence, the efficiency is 100% for the always cooperating group and 33% for the never cooperating group.

Your output should strictly be a prediction value with integer only (e.g., 33% should output 33 and nothing else)."""

def make_predict_prompt(config, augmented_text=''):
        closing =   f"""Now, predict the efficiency of the game below when punishment is to be enabled.
### Game Information ###
{config}

You predict that enabling punishment will cause the efficiency percentage to change to (output should be an integer and nothing else):
"""
        return augmented_text + closing
        
def make_config(cd): # cd: configuration dictionary
    game_structure = f"""
***The efficiency of this game with punishment disabled was:*** {int(round(100 * cd['efficiency_np'], 0))}%

[CONFIGURATION]

*** Game Structure ***
Number of players: {int(cd['CONFIG_playerCount'])}
Number of rounds: {int(cd['CONFIG_numRounds'])}
Is chat enabled among players?: {bool(cd['CONFIG_chat'])}
Is the contribution "all or nothing" i.e., binary instead of continuous?: {bool(cd['CONFIG_allOrNothing'])}
Is contribution the default i.e., does each player's endowment start in the public fund for them to opt-out?: {bool(cd['CONFIG_defaultContribProp'])}

*** Monetary Stakes ***
Marginal per capita return (MPCR): {cd['CONFIG_MPCR']}

*** Peer Incentives *** 
    """

    punishment = f"""
Punishment cost to impose a single unit of punishment: {int(cd['CONFIG_punishmentCost'])} coin(s)
Punishment impact (number of coins deducted from the punished player per coin spent punishing): {float(cd['CONFIG_punishmentTech'])}
    """
        # should it be punishment Tech? No magnitude is fine
    reward = f"""
Reward cost to grant a single unit of reward: {int(cd['CONFIG_rewardCost'])} coin(s)
Reward impact (the coins awarded to a player per coin spent rewarding): {float(cd['CONFIG_rewardTech'])}
    """

    information_display = f"""
*** Information Display ***
Is the number of rounds known to players (do they know when the game ends)?: {bool(cd['CONFIG_showNRounds'])}
Are peer outcomes shown (do players know how much their peers gained at the end of each round)?: {bool(cd['CONFIG_showOtherSummaries'])}"""

    information_punishment = f"""
When a player is punished/rewarded, are the punishers/rewarders known?: {bool(cd['CONFIG_showPunishmentId'])}
    """


    no_reward = f"""
Reward mechanism is not enabled.
    """

    
    if cd['CONFIG_rewardExists'] == True:
        return game_structure + punishment + reward + information_display + information_punishment
    else:
        return game_structure + punishment + no_reward + information_display + information_punishment

In [ ]:
def append_prompt(dataframe, augmented_text=''):
    messages = [ make_predict_prompt( make_config(row), augmented_text=augmented_text ) for i, row in dataframe.iterrows() ]
    #dataframe['prompt_'+column] = messages
    return messages

prompt_rag = """You have access to the files of academic papers discussing the effect of punishment on cooperative games.
Make predictions based faithfully on what you can learn from the papers' integrated findings.
"""

prompts_rag = append_prompt(df_pgg, prompt_rag)

In [ ]:
print(prompts_rag[0])

In [ ]:
def create_prediction_batch_json( model='gpt-4.1-2025-04-14', prompts = prompts_rag):
    requests = list()

    for n, p in enumerate(prompts):
        request = {
            "custom_id": f"ALL/Q{n+1}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": model,
                "input": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": p}
                ],
            "tools": [
                {
                    "type": "file_search",
                    "vector_store_ids": ["vs_68e867cb856881919afaf916060dcea8"],
                    "max_num_results": 50
                    }
                    ],
            "include": ["file_search_call.results", "message.output_text.logprobs"],
                "temperature": 0,
                #"logprobs": True,
                "top_logprobs": 20,
                #"max_output_tokens": 16
            }
        }
        requests.append(request)
    return requests


In [ ]:
df_elig

In [ ]:
df_elig['file_id']

In [ ]:
def create_filtered_prediction_batch_json( model='gpt-4.1-2025-04-14', prompts = prompts_rag):
    requests = list()
    for k, v in coll_map.items():
        file_ids = df_elig['file_id'].iloc[v]  # file id list
        
        for n, p in enumerate(prompts):
            request = {
                "custom_id": f"{k}/Q{n+1}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": model,
                    "input": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": p}
                    ],
                "tools": [
                    {
                        "type": "file_search",
                        "vector_store_ids": ["vs_68e867cb856881919afaf916060dcea8"],
                        "filters": {
                            "type": "in",
                            "key": "fileId",
                            "value": list(file_ids)
                        },
                        "max_num_results": 50
                        }
                        ],
                "include": ["file_search_call.results", "message.output_text.logprobs"],
                    "temperature": 0,
                    "top_logprobs": 20,
                }
            }
            requests.append(request)
    return requests


In [ ]:
with open(f"OpenAI_batch_input/prediction_251108_RAG_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for i, entry in enumerate(create_prediction_batch_json()):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [ ]:
with open(f"OpenAI_batch_input/prediction_251110_RAG_41.jsonl", "w", encoding="utf-8") as jsonl_file:
    for i, entry in enumerate(create_filtered_prediction_batch_json()):
        jsonl_file.write(json.dumps(entry, ensure_ascii=False) + "\n")